# The normalized GNM calculation

This example checks the implemented Kirchhoff/pseudoinverse relationship on
locally bundled TcTIM. It is a numerical contract demonstration, not a
reproduction of a published experimental benchmark.

In [ ]:
from importlib.resources import as_file, files
import numpy as np
import molsysmt as msm
from elastnetmt import AnisotropicNetworkModel, GaussianNetworkModel, pyunitwizard as puw

resource = files("molsysmt").joinpath("data/pdb/1tcd.pdb")
with as_file(resource) as path:
    molecular_system = msm.convert(str(path), to_form="molsysmt.MolSys")

In [ ]:
gnm = GaussianNetworkModel(molecular_system, engine="vectorized")
values = gnm.get_eigenvalues()
vectors = gnm.get_eigenvectors()
matrix = gnm.kirchhoff_matrix
np.testing.assert_allclose(matrix, matrix.T)
np.testing.assert_allclose(matrix.sum(axis=1), 0)
assert values[0] == 0 and np.all(values[1:] > 0)
np.testing.assert_allclose(matrix @ vectors, vectors * values, atol=1e-10)
np.testing.assert_allclose(vectors.T @ vectors, np.eye(gnm.n_nodes), atol=1e-10)

For unit springs, the raw node fluctuation is the diagonal of the
Kirchhoff pseudoinverse. Experimental calibration sets its B-factor scale.

In [ ]:
raw = gnm.get_b_factors()
independent = np.diag(np.linalg.pinv(matrix, hermitian=True))
np.testing.assert_allclose(raw, independent, rtol=1e-9, atol=1e-10)
assert gnm.scaling_factor == 1.0

The graph must be connected, giving one rigid zero mode. Disconnected
networks raise `DegenerateNetworkError`. Significant negative eigenvalues
raise `InvalidSpectrumError`; taking their absolute value would conceal a
failed decomposition. See the [GNM API](../../api/users/models.rst).